# Deduplication Audit Dashboard
## Pipeline Step 4: Trace how duplicates are identified at every stage

| Stage | Folder | Content |
|---|---|---|
| 0 | `data/01_raw` | raw rows |
| 1 | `data/02_by_category` | category partitions |
| 2 | `data/03_clusters` | `Material name` clustering (norm name, `cluster_id`, `new_master_code`, score) |
| 3 | `data/04_curated` | curated per-category rows |
| 4 | `data/05_unique` | one row per `new_master_code` |

Clustering uses `Material name` only; `Colour` plays no part. Sections 1-6 follow the pipeline steps. Run `01` -> `03` first.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from rapidfuzz import process as rf_process, fuzz as rf_fuzz
    from rapidfuzz.distance import Levenshtein as rf_lev
    def sim(a, b):
        return 1.0 if a == b else max(rf_lev.normalized_similarity(a, b), rf_fuzz.token_sort_ratio(a, b) / 100.0)
except ImportError:
    import difflib
    def sim(a, b):
        return 1.0 if a == b else difflib.SequenceMatcher(None, a, b).ratio()

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})
PALETTE = ["#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974", "#64B5CD", "#937860"]
pd.set_option("display.max_colwidth", 70, "display.width", 200)
thousands = FuncFormatter(lambda v, _: f"{int(v):,}")

CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "notebook"] else CURRENT_DIR
RAW_DIR, PART_DIR = ROOT / "data" / "01_raw", ROOT / "data" / "02_by_category"
CLU_DIR, CUR_DIR, UNI_DIR = ROOT / "data" / "03_clusters", ROOT / "data" / "04_curated", ROOT / "data" / "05_unique"

rules = ROOT / "config" / "splink_rules.yaml"
THRESHOLD = float((yaml.safe_load(open(rules, encoding="utf-8")) or {}).get("match_threshold", 0.85)) if rules.exists() else 0.85
print(f"[CONFIG] similarity threshold = {THRESHOLD}")

In [ ]:
# -------------------------------------------------------------------------
# Load every stage (each category kept separate)
# -------------------------------------------------------------------------
raw_files = sorted(RAW_DIR.glob("*.csv"))
raw_rows = sum(len(pd.read_csv(f, usecols=[0])) for f in raw_files)

slugs = [f.stem.replace("curated_", "", 1) for f in sorted(CUR_DIR.glob("curated_*.csv"))]
if not slugs:
    raise FileNotFoundError("No curated files found. Run notebooks 01-03 first.")

part, clu, cur, uni = {}, {}, {}, {}
for s in slugs:
    part[s] = pd.read_csv(PART_DIR / f"category_{s}.csv", dtype=str, keep_default_na=False)
    clu[s] = pd.read_csv(CLU_DIR / f"clustered_{s}.csv", dtype=str, keep_default_na=False)
    cur[s] = pd.read_csv(CUR_DIR / f"curated_{s}.csv", dtype=str, keep_default_na=False)
    uni[s] = pd.read_csv(UNI_DIR / f"unique_master_{s}.csv", dtype=str, keep_default_na=False)
    clu[s]["similarity_score"] = clu[s]["similarity_score"].astype(float)
    clu[s]["cluster_record_count"] = clu[s]["cluster_record_count"].astype(int)
    clu[s]["cluster_id"] = clu[s]["cluster_id"].astype(int)

label = {s: s.replace("_", " ").title() for s in slugs}
funnel = pd.DataFrame({
    "Category": [label[s] for s in slugs],
    "Partition rows": [len(part[s]) for s in slugs],
    "Distinct raw names": [part[s]["Material name"].nunique() for s in slugs],
    "Distinct normalized names": [clu[s]["norm_material_name"].nunique() for s in slugs],
    "Clusters": [clu[s]["cluster_id"].nunique() for s in slugs],
    "Unique masters (05_unique)": [len(uni[s]) for s in slugs],
}, index=slugs)
print(f"[LOAD] raw rows: {raw_rows:,} | partition rows: {funnel['Partition rows'].sum():,} | masters: {funnel.iloc[:, -1].sum():,}")
display(funnel)

## 1. Data funnel & compression

In [ ]:
totals = [("Raw rows (01_raw)", raw_rows),
          ("Category partitions (02)", funnel["Partition rows"].sum()),
          ("Distinct raw names", funnel["Distinct raw names"].sum()),
          ("Distinct normalized names", funnel["Distinct normalized names"].sum()),
          ("Clusters (03)", funnel["Clusters"].sum()),
          ("Unique masters (05)", funnel["Unique masters (05_unique)"].sum())]
names, vals = zip(*totals)

fig, ax = plt.subplots(1, 2, figsize=(17, 5))
y = np.arange(len(vals))[::-1]
ax[0].barh(y, vals, color=PALETTE[0], height=0.65)
ax[0].set_xscale("log"); ax[0].set_yticks(y); ax[0].set_yticklabels(names)
for yi, v, prev in zip(y, vals, [None] + list(vals[:-1])):
    note = f"{v:,}" + (f"  ({(1 - v / prev) * 100:.1f}% drop)" if prev else "")
    ax[0].text(v * 1.1, yi, note, va="center", fontsize=9)
ax[0].set_xlim(right=max(vals) * 40); ax[0].set_title("Overall funnel (log scale)")

w = 0.2; x = np.arange(len(funnel))
for i, col in enumerate(["Partition rows", "Distinct raw names", "Distinct normalized names", "Unique masters (05_unique)"]):
    ax[1].bar(x + (i - 1.5) * w, funnel[col], w, label=col, color=PALETTE[i])
ax[1].set_yscale("log"); ax[1].set_xticks(x); ax[1].set_xticklabels(funnel["Category"], rotation=25)
ax[1].set_title("Per-category shrinkage (log scale)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

red = funnel[["Category"]].copy()
red["Rows -> masters reduction %"] = ((1 - funnel["Unique masters (05_unique)"] / funnel["Partition rows"]) * 100).round(3)
red["Raw names -> masters reduction %"] = ((1 - funnel["Unique masters (05_unique)"] / funnel["Distinct raw names"]) * 100).round(2)
overall = (1 - funnel["Unique masters (05_unique)"].sum() / raw_rows) * 100
print(f"Overall reduction raw rows -> unique masters: {overall:.3f}%")
display(red)

## 2. Step 1 - Impact of category isolation

In [ ]:
cat_tbl = funnel[["Category", "Partition rows", "Distinct raw names"]].copy()
cat_tbl["% of rows"] = (cat_tbl["Partition rows"] / cat_tbl["Partition rows"].sum() * 100).round(2)
cat_tbl["Missing / uncategorized rows"] = [
    int((part[s].get("Category", pd.Series(dtype=str)).str.strip() == "").sum()) if "Category" in part[s] else 0 for s in slugs]
cat_tbl.loc["uncategorized", "Missing / uncategorized rows"] = len(part["uncategorized"]) if "uncategorized" in part else 0
display(cat_tbl)

# Which normalized names exist in more than one category, and which cross-category pairs would
# have merged (>= threshold) if everything had been compared together?
named = [s for s in slugs if s != "uncategorized"]
cat_names = {s: sorted(clu[s]["norm_material_name"].unique()) for s in named}
rows = []
for i, a in enumerate(named):
    for b in named[i + 1:]:
        na, nb_ = cat_names[a], cat_names[b]
        exact = len(set(na) & set(nb_))
        if "rf_process" in globals():
            m = rf_process.cdist(na, nb_, scorer=rf_lev.normalized_similarity, workers=-1)
            fuzzy = int((m >= THRESHOLD).sum())
        else:
            fuzzy = sum(sim(x, y) >= THRESHOLD for x in na for y in nb_)
        rows.append({"Category A": label[a], "Category B": label[b], "Exact shared names": exact,
                     "Cross-category pairs >= threshold": fuzzy})
leak = pd.DataFrame(rows)
print("Name pairs that would have been merged across categories without Step 1 partitioning:")
display(leak)
print(f"Total avoided cross-category pairs >= {THRESHOLD}: {int(leak['Cross-category pairs >= threshold'].sum()):,}")

# Concrete examples
ex = []
for i, a in enumerate(named):
    for b in named[i + 1:]:
        for x in cat_names[a]:
            for y in cat_names[b]:
                if x[:4] == y[:4] and sim(x, y) >= 0.6 and len(ex) < 8:
                    ex.append({"Category A": label[a], "Name A": x, "Category B": label[b], "Name B": y, "Similarity": round(sim(x, y), 3)})
if ex:
    print("\nLookalike names across categories (kept apart by partitioning):")
    display(pd.DataFrame(ex))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
bars = ax[0].bar(cat_tbl["Category"], cat_tbl["Partition rows"], color=PALETTE[:len(cat_tbl)]); ax[0].bar_label(bars, fmt="{:,.0f}")
ax[0].set_title("Records per category"); ax[0].tick_params(axis="x", rotation=25); ax[0].yaxis.set_major_formatter(thousands)
bars = ax[1].bar(cat_tbl["Category"], cat_tbl["Distinct raw names"], color=PALETTE[:len(cat_tbl)]); ax[1].bar_label(bars)
ax[1].set_title("Distinct raw names per category"); ax[1].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

## 3. Step 2 - Text normalization (exact collapsing)

In [ ]:
SYMBOLS = {"&": " and ", "%": " percent ", "+": " plus ", "@": " at ", "#": " number ",
           "/": " ", "-": " ", "_": " ", "\u00d7": " x "}

def norm_stage(text, sort=False, strip=True):
    t = str(text)
    if strip:
        t = t.lower()
        for k, v in SYMBOLS.items():
            t = t.replace(k, v)
        t = re.sub(r"(?<=\d)x(?=\d)", " ", t)
        t = re.sub(r"[^\w\s]", " ", t)
    toks = t.split()
    return " ".join(sorted(toks) if sort else toks)

rows = []
for s in slugs:
    raw_n = part[s]["Material name"].unique()
    exact = len(raw_n)
    lower_punct = len({norm_stage(r) for r in raw_n})
    full = len({norm_stage(r, sort=True) for r in raw_n})
    rows.append({"Category": label[s], "Distinct raw names": exact, "After lowercase + punctuation": lower_punct,
                 "After token sorting (= norm name)": full,
                 "Collapsed by case/punct.": exact - lower_punct, "Collapsed by token sort": lower_punct - full,
                 "Total exact collapse": exact - full})
norm_impact = pd.DataFrame(rows)
display(norm_impact)
tot = norm_impact[["Distinct raw names", "Collapsed by case/punct.", "Collapsed by token sort"]].sum()
print(f"Exact duplicates removed purely by normalization: {int(tot.iloc[1] + tot.iloc[2]):,} of {int(tot.iloc[0]):,} distinct raw names "
      f"({(tot.iloc[1] + tot.iloc[2]) / tot.iloc[0] * 100:.1f}%); token re-ordering alone: {int(tot.iloc[2]):,}")

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(norm_impact))
ax.bar(x, norm_impact["After token sorting (= norm name)"], color=PALETTE[1], label="Remaining after normalization")
ax.bar(x, norm_impact["Collapsed by case/punct."], bottom=norm_impact["After token sorting (= norm name)"], color=PALETTE[4], label="Collapsed: case/punctuation")
ax.bar(x, norm_impact["Collapsed by token sort"], bottom=norm_impact["After lowercase + punctuation"], color=PALETTE[2], label="Collapsed: token sort")
ax.set_xticks(x); ax.set_xticklabels(norm_impact["Category"], rotation=25); ax.legend(); ax.set_title("Distinct raw names: what normalization removed")
plt.tight_layout(); plt.show()

# Sample: token reordering immediately merged several variants into one entity
samples = []
for s in slugs:
    d = part[s].drop_duplicates("Material name").copy()
    d["lower_punct"] = d["Material name"].map(norm_stage)
    d["norm_material_name"] = d["Material name"].map(lambda t: norm_stage(t, sort=True))
    grp = d.groupby("norm_material_name").agg(variants=("Material name", lambda x: " | ".join(x)),
                                              n_variants=("Material name", "nunique"),
                                              n_lower_forms=("lower_punct", "nunique"))
    grp = grp[grp["n_lower_forms"] > 1].sort_values("n_variants", ascending=False).head(5).reset_index()
    grp.insert(0, "Category", label[s]); samples.append(grp)
sample_tbl = pd.concat(samples, ignore_index=True)
print("\nToken re-ordering collapsed these variants into a single entity before any fuzzy matching:")
display(sample_tbl[["Category", "norm_material_name", "n_variants", "variants"]])

## 4. Step 3 - Fuzzy matching & clustering distribution

In [ ]:
# Variant level: one record per (category, cluster, normalized name)
var = pd.concat([
    c.groupby(["cluster_id", "new_master_code", "norm_material_name"]).agg(
        score=("similarity_score", "min"), rows=("Material name", "size"),
        display_name=("Material name", lambda x: x.value_counts().index[0])).reset_index().assign(Category=label[s])
    for s, c in clu.items()], ignore_index=True)
cl = var.groupby(["Category", "cluster_id", "new_master_code"]).agg(variants=("norm_material_name", "nunique"), rows=("rows", "sum")).reset_index()

fuzzy = var[var["score"] < 1.0]
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
if len(fuzzy):
    ax[0].hist(fuzzy["score"], bins=np.linspace(THRESHOLD - 0.02, 1.0, 26), color=PALETTE[2], alpha=0.8, density=True)
    try:
        from scipy.stats import gaussian_kde
        xs = np.linspace(THRESHOLD, 1.0, 200); ax[0].plot(xs, gaussian_kde(fuzzy["score"])(xs), color="black", lw=1.5, label="KDE")
        ax[0].legend()
    except Exception:
        pass
ax[0].axvline(THRESHOLD, ls="--", color="grey"); ax[0].set_title(f"Fuzzy-matched variants: score vs representative (n={len(fuzzy):,})")
ax[0].set_xlabel("similarity_score")

bins = [0, 1, 5, 10, 10 ** 9]; lbls = ["1 (singleton)", "2-5", "6-10", "10+"]
cl["variant_bucket"] = pd.cut(cl["variants"], bins=bins, labels=lbls)
vb = cl["variant_bucket"].value_counts().reindex(lbls)
bars = ax[1].bar(lbls, vb.values, color=PALETTE[0]); ax[1].bar_label(bars)
ax[1].set_title("Clusters by # distinct name variants"); ax[1].set_xlabel("distinct normalized names in cluster")

rb = ["<=10", "11-100", "101-1000", "1000+"]
cl["row_bucket"] = pd.cut(cl["rows"], bins=[0, 10, 100, 1000, 10 ** 9], labels=rb)
rv = cl["row_bucket"].value_counts().reindex(rb)
bars = ax[2].bar(rb, rv.values, color=PALETTE[3]); ax[2].bar_label(bars)
ax[2].set_title("Clusters by # raw records"); ax[2].set_xlabel("raw records in cluster")
plt.tight_layout(); plt.show()

print("Top 5 largest clusters per category:")
top5 = (cl.sort_values("rows", ascending=False).groupby("Category").head(5)
        .merge(var.sort_values("rows", ascending=False).drop_duplicates("new_master_code")[["Category", "new_master_code", "display_name"]],
               on=["Category", "new_master_code"], how="left")
        .rename(columns={"display_name": "canonical_name", "rows": "raw_records"})
        .sort_values(["Category", "raw_records"], ascending=[True, False]))
display(top5[["Category", "new_master_code", "canonical_name", "variants", "raw_records"]].reset_index(drop=True))

## 5. Step 4 - Transitive chaining & edge-case review

In [ ]:
# 5a. Borderline matches
border = var[(var["score"] >= THRESHOLD) & (var["score"] < 1.0)].sort_values("score")
rep = var.sort_values("rows", ascending=False).drop_duplicates("new_master_code").set_index(["Category", "new_master_code"])["norm_material_name"]
border = border.assign(representative=[rep.get((c, m), "") for c, m in zip(border["Category"], border["new_master_code"])])
n_border = int((border["score"] < THRESHOLD + 0.03).sum())
print(f"Borderline variants (score {THRESHOLD:.2f}-{THRESHOLD + 0.03:.2f}): {n_border:,} of {len(border):,} fuzzy-matched variants")
display(border[["Category", "new_master_code", "norm_material_name", "representative", "score", "rows"]].head(15).reset_index(drop=True))

# 5b. Transitive chaining: members joined via a chain but not directly similar to each other
chain_rows = []
for (cat, code_), g in var[var["new_master_code"].isin(cl.loc[cl["variants"] > 1, "new_master_code"])].groupby(["Category", "new_master_code"]):
    nm = g["norm_material_name"].tolist()
    if len(nm) > 60:
        nm = nm[:60]
    worst = min((sim(a, b), a, b) for i, a in enumerate(nm) for b in nm[i + 1:])
    chain_rows.append({"Category": cat, "new_master_code": code_, "variants": len(g), "min_pairwise_similarity": round(worst[0], 3),
                       "least_similar_pair": f"{worst[1]}  <->  {worst[2]}"})
chain = pd.DataFrame(chain_rows)
if len(chain):
    chained = chain[chain["min_pairwise_similarity"] < THRESHOLD].sort_values("min_pairwise_similarity")
    print(f"\nMulti-variant clusters: {len(chain):,} | with transitive chaining (some pair < {THRESHOLD}): {len(chained):,}")
    display(chained.head(15).reset_index(drop=True) if len(chained) else "No transitive chaining detected.")
else:
    print("No multi-variant clusters.")

# 5c. Side-by-side grouping check
multi = cl[cl["variants"] > 1].sort_values("variants", ascending=False).head(6)
for _, r in multi.iterrows():
    g = var[(var["Category"] == r["Category"]) & (var["new_master_code"] == r["new_master_code"])].sort_values("rows", ascending=False)
    print(f"\n{r['Category']} | {r['new_master_code']} | {r['variants']} variants | {r['rows']:,} records")
    display(g[["display_name", "norm_material_name", "score", "rows"]].rename(columns={"display_name": "original name (most common)", "rows": "records"}).head(8).reset_index(drop=True))

## 6. Step 5 - Final master state (`05_unique`)

In [ ]:
final = pd.DataFrame({
    "Category": [label[s] for s in slugs],
    "Original raw record count": [len(part[s]) for s in slugs],
    "Final unique master codes": [len(uni[s]) for s in slugs],
}, index=slugs)
final["Duplication reduction %"] = ((1 - final["Final unique master codes"] / final["Original raw record count"]) * 100).round(3)
final["True duplicates resolved"] = final["Original raw record count"] - final["Final unique master codes"]
total = final.sum(numeric_only=True)
total["Duplication reduction %"] = round((1 - total["Final unique master codes"] / total["Original raw record count"]) * 100, 3)
final.loc["TOTAL"] = ["ALL"] + total.tolist()
display(final.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 4))
f = final.drop("TOTAL"); x = np.arange(len(f))
ax.bar(x - 0.2, f["Original raw record count"], 0.4, label="Before (raw records)", color=PALETTE[0])
ax.bar(x + 0.2, f["Final unique master codes"], 0.4, label="After (unique masters)", color=PALETTE[1])
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(f["Category"], rotation=25); ax.legend(); ax.set_title("Before vs after (log scale)")
plt.tight_layout(); plt.show()

# Integrity checks
checks = []
for s in slugs:
    u = uni[s]
    checks.append({"Category": label[s],
                   "1 row per new_master_code": u["new_master_code"].is_unique,
                   "no blank codes": bool((u["new_master_code"].str.strip() != "").all()),
                   "master set == clustered codes": set(u["new_master_code"]) == set(clu[s]["new_master_code"]),
                   "cluster counts sum to rows": int(u["cluster_record_count"].astype(int).sum()) == len(part[s]) == len(cur[s]) == len(clu[s]),
                   "exact 6-column schema": list(u.columns) == ["new_master_code", "norm_material_name", "Material name", "Category", "cluster_id", "cluster_record_count"]})
chk = pd.DataFrame(checks)
display(chk)
assert chk.drop(columns="Category").all().all(), "Integrity check failed - see table above"
print("All integrity checks passed: every 05_unique file has strictly one row per new_master_code.")

## 7. Interactive category explorer (optional)
Requires `ipywidgets`; otherwise call `explore('fabric')` directly.

In [ ]:
def explore(slug="fabric", top=10):
    c = clu[slug]
    summary = (c.groupby(["new_master_code", "cluster_id"]).agg(records=("Material name", "size"), variants=("norm_material_name", "nunique"),
               min_score=("similarity_score", "min"), canonical=("Material name", lambda x: x.value_counts().index[0]))
               .reset_index().sort_values("records", ascending=False).head(top))
    display(summary)
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.barh(summary["new_master_code"][::-1], summary["records"][::-1], color=PALETTE[0]); ax.set_title(f"{label[slug]}: top {top} clusters by records")
    plt.tight_layout(); plt.show()

try:
    import ipywidgets as widgets
    widgets.interact(explore, slug=widgets.Dropdown(options=slugs, value=slugs[0]), top=widgets.IntSlider(value=10, min=3, max=30))
except ImportError:
    explore(slugs[0])